# Fine-Tuning GPT-2 for SOP-Driven Text Generation
**Author:** Javohirbek Xatamov

### 1. Install Dependencies
We need the Hugging Face ecosystem and evaluation metrics.

In [1]:
!pip install -q transformers[torch] datasets evaluate rouge_score

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 8.6 MB/s eta 0:00:00


### 2. Load SOP Dataset
Please upload your `SOP.txt` file using the button below.

In [3]:
from google.colab import files
import os

# Find the uploaded file even if it was renamed by Colab (e.g., SOP (1).txt)
target_file = 'SOP.txt'
uploaded_files = list(uploaded.keys())

if any('SOP' in f and f.endswith('.txt') for f in uploaded_files):
    # Get the most recent SOP file
    filename = [f for f in uploaded_files if 'SOP' in f and f.endswith('.txt')][-1]
    with open(filename, 'r', encoding='utf-8') as f:
        sop_text = f.read().strip()
    print(f'Successfully loaded {filename}')
    print(f'File contains {len(sop_text.split())} words.')
else:
    print('Warning: SOP.txt not found. Please upload the file.')

Successfully loaded SOP (1).txt
File contains 1544 words.


### 3. Tokenization
We will use the GPT-2 tokenizer to break the SOP into blocks of 128 tokens for training.

In [13]:
from transformers import AutoTokenizer, DataCollatorForLanguageModeling
from datasets import Dataset

model_checkpoint = "gpt2"
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)
tokenizer.pad_token = tokenizer.eos_token

# Tokenize the entire text
tokens = tokenizer(sop_text, truncation=False)["input_ids"]

# Fix: Ensure all blocks are exactly 128 tokens by discarding the final incomplete chunk
block_size = 128
chunks = [tokens[i : i + block_size] for i in range(0, len(tokens) - block_size + 1, block_size)]

# Create a HuggingFace Dataset
dataset_dict = {"input_ids": chunks, "labels": chunks.copy()}
train_dataset = Dataset.from_dict(dataset_dict)

print(f"Prepared {len(train_dataset)} training blocks of length {block_size}.")

Token indices sequence length is longer than the specified maximum sequence length for this model (1985 > 1024). Running this sequence through the model will result in indexing errors


Prepared 15 training blocks of length 128.


### 4. Model Loading and Setup
Loading the pre-trained GPT-2 model and configuring the embeddings.

In [5]:
from transformers import GPT2LMHeadModel, TrainingArguments, Trainer

model = GPT2LMHeadModel.from_pretrained(model_checkpoint)
model.resize_token_embeddings(len(tokenizer))

print("Model loaded and embedding layer resized.")

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

Model loaded and embedding layer resized.


### 5. Fine-Tuning Process
Setting up the training hyperparameters and running the trainer.

### 6. Text Generation and 7. ROUGE-L Evaluation
Generating samples based on your name and evaluating them against the original SOP.

In [16]:
import evaluate
import torch
from transformers import GPT2LMHeadModel

# Load the fine-tuned model for inference
fine_tuned_model = GPT2LMHeadModel.from_pretrained('./gpt2_sop_model/')
fine_tuned_model.to('cuda' if torch.cuda.is_available() else 'cpu')
fine_tuned_model.eval()

# Setup generation
prompt = "Javohirbek Xatamov's academic interests include"
input_ids = tokenizer.encode(prompt, return_tensors='pt').to(fine_tuned_model.device)

# Generate 3 samples using the fine-tuned model
outputs = fine_tuned_model.generate(
    input_ids,
    max_length=150,
    num_return_sequences=3,
    no_repeat_ngram_size=2,
    do_sample=True,
    top_k=50,
    top_p=0.95,
    temperature=0.8,
    pad_token_id=tokenizer.eos_token_id
)

generated_texts = [tokenizer.decode(output, skip_special_tokens=True) for output in outputs]

# 8. ROUGE Evaluation
rouge = evaluate.load('rouge')
print(f"Prompt: {prompt}\n" + "="*30)

rouge_scores = []
for i, text in enumerate(generated_texts):
    results = rouge.compute(predictions=[text], references=[sop_text])
    score = results['rougeL']
    rouge_scores.append(score)
    print(f"\nSample {i+1}:\n{text}")
    print(f"ROUGE-L Score: {score:.4f}")
    print("-"*30)

print(f"\nAverage ROUGE-L Score: {sum(rouge_scores)/len(rouge_scores):.4f}")

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

Prompt: Javohirbek Xatamov's academic interests include

Sample 1:
Javohirbek Xatamov's academic interests include:

Research Methods
: Data Analysis
, Statistics, Machine Learning, and Complexity
My current research interests involve: Computer Science, Optimization of Statistical Models, Logistic Regression Models in Computer Vision, Deep Learning and Deep Convolutional Networks, Parallelism and the Analysis of Complex Databases, Information Theory, Application Theory and Data Mining, Software Engineering and Engineering, Human Factors Management, Model Management and Development, Computer Model Development and Implementation, Network Analysis and Network Security, User Experience Management for Mobile Networks and Application Development in Virtual and Offline Contexts, Introduction to Visualization and Design, Virtual Reality and Virtualization, Development for Java Web Services and Mobile Computing, Digital
ROUGE-L Score: 0.0414
------------------------------

Sample 2:
Javohirbek 

### 9. Final Output Confirmation
Checking the saved model directory to confirm successful training.

In [17]:
if os.path.exists('./gpt2_sop_model/config.json'):
    print('Model save confirmation: [SUCCESS]')
    print('You can now download the notebook as YourStudentID.ipynb.')
else:
    print('Model save confirmation: [PENDING] Please run the training cell.')

Model save confirmation: [SUCCESS]
You can now download the notebook as YourStudentID.ipynb.


In [18]:
training_args = TrainingArguments(
    output_dir="./gpt2_sop_model",
    num_train_epochs=5,
    per_device_train_batch_size=2,
    save_steps=50,
    logging_steps=10,
    learning_rate=5e-5,
    weight_decay=0.01,
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    data_collator=DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False),
)

trainer.train()
trainer.save_model("./gpt2_sop_model/")
print("Model fine-tuned and saved to ./gpt2_sop_model/")

Step,Training Loss
10,2.327984
20,1.916561
30,1.701997
40,1.653010


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model fine-tuned and saved to ./gpt2_sop_model/
